# Label Studio 프로젝트 13 — P3 Gold 데이터 한 장씩 저장

- 대상: http://203.255.15.92:48000/projects/13/data?tab=10
- **Project 13 / tab 10**에 표시되는 작업을 API로 조회하고 Task ID 순서대로 한 장씩 받습니다.
- 확인 시 프로젝트 이름은 **P3-Masking**, 작업은 **72개**이며 실제 실행 시 목록을 다시 조회합니다.
- 위에서부터 **4. 한 장씩 다운로드 및 저장**까지 실행하거나 `Run All`을 사용하세요.
- 저장 위치: `data/gold/task_<ID>/`. 이미지·마스크 배열은 현재 작업 하나만 유지합니다.
- 완료된 Annotation 중 최신 버전 하나만 사용합니다. Exposure 마스크는 `alpha > 0`입니다.
- 프로젝트 10의 Task 702/703/704, 시편 매칭표, data/0918 원본 치환은 사용하지 않습니다.
- 등록 이미지의 원래 크기와 좌표를 유지하고 학습·추론·서버 라벨 수정은 실행하지 않습니다.

인증은 `LABEL_STUDIO_REFRESH_TOKEN` 환경변수 또는 기존 `config/label_studio_project9.local.json`의 토큰을 재사용합니다. 이 인증 파일의 project_id는 다운로드 대상을 바꾸지 않습니다.

필요한 패키지 설치 (현재 프로젝트 `.venv`를 노트북 커널로 선택):
```powershell
Set-Location D:\Code\vision_data
uv pip install --python .venv/Scripts/python.exe ipykernel requests numpy pillow
```


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import parse_qs, unquote, urljoin, urlsplit
from getpass import getpass
import hashlib
import io
import json
import os
import sys

import numpy as np
import requests
from PIL import Image, ImageDraw, ImageFont

candidates = [Path.cwd(), *Path.cwd().parents, Path("D:/Code/vision_data")]
ROOT = next((p.resolve() for p in candidates if (p / "src/segmentation/label_studio.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("vision_data 저장소 안에서 노트북을 열어주세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.segmentation.label_studio import decode_brush_alpha

PROJECT_URL = "http://203.255.15.92:48000/projects/13/data?tab=10"
BASE_URL = os.environ.get("LABEL_STUDIO_URL", "http://203.255.15.92:48000/").rstrip("/") + "/"
PROJECT_ID = 13
VIEW_ID = 10  # URL의 tab=10에 해당하는 Data Manager view입니다.
TASK_IDS = None  # None: 해당 tab 전체. 일부만 받으려면 예: (802, 803)
CONFIG_PATH = ROOT / "config/label_studio_project9.local.json"
settings = json.loads(CONFIG_PATH.read_text(encoding="utf-8")) if CONFIG_PATH.is_file() else {}
REFRESH_TOKEN = os.environ.get("LABEL_STUDIO_REFRESH_TOKEN") or settings.get("refresh_token")
if not REFRESH_TOKEN:
    REFRESH_TOKEN = getpass("Label Studio refresh token: ")
LABEL_NAME = "Exposure"
CACHE_DIR = ROOT / "data/gold"
PREVIEW_WIDTH = 512
DEFAULT_OPACITY = 0.45
print(f"대상: Project {PROJECT_ID} / tab {VIEW_ID} · {BASE_URL}")
print(f"저장 폴더: {CACHE_DIR}")
# 토큰이나 settings 전체를 출력하지 않습니다. 이 셀에서는 서버에 접속하지 않습니다.


## 1. API 클라이언트

제공한 토큰은 JWT **refresh token**입니다. 실행 시 `/api/token/refresh/`에서 access token을 받고, 이후 요청에 `Authorization: Bearer ...`를 사용합니다. 만료 응답(401)이 오면 한 번 갱신합니다.

인증 헤더는 설정한 Label Studio 서버에만 전송합니다. 이 노트북은 서버 라벨을 수정하지 않습니다.

In [ ]:
class LabelStudioClient:
    def __init__(self, base_url, refresh_token, project_id, session=None):
        self.base_url = base_url.rstrip("/") + "/"
        self.project_id = int(project_id)
        self._refresh_token = refresh_token
        self._access_token = None
        self._session = session or requests.Session()

    def _url(self, reference):
        url = urljoin(self.base_url, reference)
        wanted, actual = urlsplit(self.base_url), urlsplit(url)
        if actual.scheme not in ("http", "https") or actual.username or actual.password:
            raise ValueError("지원하지 않는 이미지/API URL입니다.")
        def origin(parts):
            return parts.scheme.lower(), parts.hostname, parts.port or (443 if parts.scheme == "https" else 80)
        if origin(wanted) != origin(actual):
            raise ValueError("다른 서버로 토큰을 전송하지 않습니다. data.image 주소를 확인하세요.")
        return url

    def _request(self, method, reference, **kwargs):
        try:
            response = self._session.request(
                method, self._url(reference), timeout=(10, 90),
                allow_redirects=False, **kwargs
            )
        except requests.RequestException:
            raise RuntimeError("Label Studio 요청 실패: 서버 연결·네트워크 상태를 확인하세요.") from None
        if 300 <= response.status_code < 400:
            response.close()
            raise RuntimeError("서버가 다른 주소로 이동시켰습니다. 서버 URL 또는 이미지 접근 권한을 확인하세요.")
        return response

    def _json(self, response):
        try:
            return response.json()
        except ValueError:
            raise RuntimeError("서버 응답이 JSON 형식이 아닙니다. URL과 접근 권한을 확인하세요.") from None
        finally:
            response.close()

    def refresh_access(self):
        response = self._request(
            "POST", "api/token/refresh/",
            json={"refresh": self._refresh_token},
            headers={"Accept": "application/json"}
        )
        if response.status_code != 200:
            status = response.status_code
            response.close()
            raise RuntimeError(f"토큰 갱신 실패 (HTTP {status}). 로컬 설정의 토큰을 확인하세요.")
        token = self._json(response).get("access")
        if not isinstance(token, str) or not token:
            raise RuntimeError("토큰 갱신 응답에 access 값이 없습니다.")
        self._access_token = token

    def get(self, reference, **kwargs):
        if self._access_token is None:
            self.refresh_access()
        for attempt in range(2):
            response = self._request(
                "GET", reference,
                headers={"Authorization": f"Bearer {self._access_token}"},
                **kwargs
            )
            if response.status_code == 401 and attempt == 0:
                response.close()
                self.refresh_access()
                continue
            if response.status_code != 200:
                status = response.status_code
                response.close()
                raise RuntimeError(f"조회 실패 (HTTP {status}). 프로젝트/파일 권한을 확인하세요.")
            return response
        raise RuntimeError("인증 갱신 후에도 요청하지 못했습니다.")

    def get_json(self, reference, **kwargs):
        return self._json(self.get(reference, **kwargs))

    def view(self, view_id):
        view = self.get_json(f"api/dm/views/{int(view_id)}/")
        project = view.get("project")
        if isinstance(project, dict):
            project = project.get("id")
        if project is None or int(project) != self.project_id or int(view["id"]) != int(view_id):
            raise RuntimeError("지정한 tab이 현재 프로젝트에 속하지 않습니다.")
        return view

    def list_task_ids(self, page_size=100, view_id=None):
        task_ids, seen, page = [], set(), 1
        while True:
            params = {"project": self.project_id, "page": page, "page_size": page_size,
                      "fields": "task_only", "query": json.dumps({"ordering": ["id"]})}
            if view_id is not None:
                params["view"] = int(view_id)
            payload = self.get_json("api/tasks/", params=params)
            if isinstance(payload, list):
                batch, total = payload, None
            elif isinstance(payload, dict):
                batch = payload.get("tasks", payload.get("results"))
                total = payload.get("total", payload.get("count"))
            else:
                raise RuntimeError("예상과 다른 작업 목록 응답입니다.")
            if not isinstance(batch, list):
                raise RuntimeError("예상과 다른 작업 목록 응답입니다.")
            if not batch:
                if total is not None and len(task_ids) < int(total):
                    raise RuntimeError("작업 목록 조회 중 개수가 바뀌었거나 페이지가 누락됐습니다. 목록을 다시 받으세요.")
                break
            for task in batch:
                task_id = int(task["id"])
                if task_id in seen:
                    raise RuntimeError("페이지에 중복 Task ID가 있습니다. 작업 목록을 다시 불러오세요.")
                seen.add(task_id)
                task_ids.append(task_id)
            if total is not None:
                if len(task_ids) >= int(total):
                    break
            elif len(batch) < page_size:
                break
            page += 1
        return sorted(task_ids)

    def task(self, task_id):
        task = self.get_json(f"api/tasks/{int(task_id)}/")
        project = task.get("project")
        if isinstance(project, dict):
            project = project.get("id")
        if project is None or int(project) != self.project_id or int(task["id"]) != int(task_id):
            raise RuntimeError("Task ID 또는 프로젝트가 현재 설정과 다릅니다.")
        return task

    def image_bytes(self, reference):
        response = self.get(reference)
        try:
            return response.content
        finally:
            response.close()

    def close(self):
        self._session.close()



## 2. 최신 Annotation 해독·한 장씩 저장 함수

각 Task 폴더에 다음 파일을 저장합니다.

- `image.png`: Label Studio 등록 이미지. PNG 원본은 다운로드 바이트를 그대로 보존합니다.
- `mask_alpha.png`, `mask_alpha.npy`: 브러시의 원래 uint8 alpha 값.
- `mask.png`: Exposure=255, 배경=0인 PNG.
- `mask.npy`: Exposure=1, 배경=0인 uint8 NumPy 배열.
- `overlay.png`: 표시용 오버레이.
- `annotation.json`: 최신 완료 Annotation 하나가 들어 있는 Label Studio task JSON.
- `task.json`: API 작업 응답 전체. 다른 Annotation 버전도 보존합니다.
- `metadata.json`: Task/Project/Annotation ID, 이미지 경로·크기·해시, Exposure 픽셀 수.

같은 폴더를 다시 받으면 서버의 현재 이미지와 최신 Annotation으로 갱신합니다. PNG·NumPy·JSON을 임시 파일에 쓴 뒤 교체하며 `metadata.json`을 마지막에 저장합니다. 마스크 크기가 다르거나 다른 라벨 형식이면 해당 작업을 실패로 기록합니다.


In [ ]:
def select_latest_annotation(task):
    annotations = [a for a in task.get("annotations", []) if not a.get("was_cancelled", False)]
    if not annotations:
        raise ValueError(f"Task {task['id']}: 완료된 Annotation이 없습니다.")
    def order(a):
        timestamp = a.get("updated_at") or a.get("created_at")
        if not timestamp:
            raise ValueError("Annotation 수정 시간이 없어 최신 버전을 결정할 수 없습니다.")
        when = datetime.fromisoformat(timestamp.replace("Z", "+00:00"))
        if when.tzinfo is None:
            when = when.replace(tzinfo=timezone.utc)
        return when, int(a["id"])
    return max(annotations, key=order)



def annotation_alpha(annotation, width, height, label="Exposure"):
    alpha = np.zeros((height, width), dtype=np.uint8)
    regions = [r for r in annotation.get("result", [])
               if r.get("type") == "brushlabels" and label in r.get("value", {}).get("brushlabels", [])]
    for region in regions:
        decoded = decode_brush_alpha(region)
        if decoded.shape != alpha.shape:
            raise ValueError(f"등록 이미지 {width}×{height}와 마스크 {decoded.shape[::-1]}의 크기가 다릅니다.")
        alpha = np.maximum(alpha, decoded)
    if not regions and annotation.get("result"):
        raise ValueError(f"최신 Annotation에 {label!r} brushlabels가 없습니다.")
    # 비어 있는 완료 Annotation은 명시적인 빈 마스크로 표시합니다.
    return alpha



def overlay_pixels(rgb, mask, opacity=0.45, color=(255, 70, 70)):
    if rgb.shape[:2] != mask.shape or not 0 <= opacity <= 1:
        raise ValueError("오버레이 크기 또는 투명도를 확인하세요.")
    result = rgb.copy()
    result[mask] = np.rint(
        (1 - opacity) * rgb[mask].astype(np.float32)
        + opacity * np.asarray(color, dtype=np.float32)
    ).astype(np.uint8)
    return result



def write_json(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)



def write_bytes(path, blob):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_bytes(blob)
    temporary.replace(path)


def write_png(path, pixels):
    temporary = path.with_suffix(path.suffix + ".tmp")
    Image.fromarray(pixels).save(temporary, format="PNG")
    temporary.replace(path)


def write_numpy(path, pixels):
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as stream:
        np.save(stream, pixels, allow_pickle=False)
    temporary.replace(path)


def download_one(client, task_id, output_root=CACHE_DIR, label=LABEL_NAME):
    task = client.task(task_id)
    annotation = select_latest_annotation(task)
    reference = task.get("data", {}).get("image")
    if not isinstance(reference, str) or not reference:
        raise ValueError(f"Task {task_id}: data.image가 없습니다.")
    blob = client.image_bytes(reference)
    try:
        with Image.open(io.BytesIO(blob)) as loaded:
            loaded.load()
            image_format = loaded.format
            rgb = np.array(loaded.convert("RGB"))
    except (OSError, ValueError):
        raise ValueError("다운로드 응답을 이미지로 읽지 못했습니다. 이미지 접근 권한을 확인하세요.") from None
    height, width = rgb.shape[:2]
    alpha = annotation_alpha(annotation, width, height, label)
    mask = alpha > 0
    folder = Path(output_root) / f"task_{int(task_id)}"
    previous_metadata = folder / "metadata.json"
    if previous_metadata.is_file():
        previous = json.loads(previous_metadata.read_text(encoding="utf-8"))
        if int(previous["project_id"]) != int(client.project_id):
            raise ValueError("같은 Task 폴더에 다른 프로젝트의 파일이 있습니다. 별도 저장 폴더를 지정하세요.")
    folder.mkdir(parents=True, exist_ok=True)
    if image_format == "PNG":
        write_bytes(folder / "image.png", blob)
    else:
        write_png(folder / "image.png", rgb)
        write_bytes(folder / "source_image.bin", blob)
    write_png(folder / "mask_alpha.png", alpha)
    write_png(folder / "mask.png", mask.astype(np.uint8) * 255)
    write_numpy(folder / "mask_alpha.npy", alpha)
    write_numpy(folder / "mask.npy", mask.astype(np.uint8))
    write_png(folder / "overlay.png", overlay_pixels(rgb, mask, DEFAULT_OPACITY))
    selected_task = {"id": int(task_id), "project": int(client.project_id),
                     "data": task["data"], "annotations": [annotation]}
    write_json(folder / "annotation.json", selected_task)
    write_json(folder / "task.json", task)
    source_path = parse_qs(urlsplit(reference).query).get("d", [unquote(urlsplit(reference).path)])[0]
    filenames = ("image.png", "mask_alpha.png", "mask.png", "mask_alpha.npy", "mask.npy",
                 "overlay.png", "annotation.json", "task.json")
    metadata = {
        "status": "completed", "task_id": int(task_id), "project_id": int(client.project_id),
        "annotation_id": int(annotation["id"]), "annotation_updated_at": annotation.get("updated_at"),
        "image_reference": reference, "source_filename": Path(source_path).name,
        "width": width, "height": height, "image_source": "label_studio_registered",
        "label": label, "mask_rule": "alpha > 0", "mask_png_values": [0, 255], "mask_npy_values": [0, 1],
        "exposure_pixels": int(mask.sum()), "empty_completed_annotation": not bool(annotation.get("result")),
        "download_sha256": hashlib.sha256(blob).hexdigest(),
        "image_sha256": hashlib.sha256((folder / "image.png").read_bytes()).hexdigest(),
        "annotation_sha256": hashlib.sha256(json.dumps(annotation, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest(),
        "files_sha256": {name: hashlib.sha256((folder / name).read_bytes()).hexdigest() for name in filenames},
        "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    write_json(folder / "metadata.json", metadata)
    return {"rgb": rgb, "alpha": alpha, "mask": mask, "folder": folder, "metadata": metadata}


def preview_image(record, opacity=DEFAULT_OPACITY, width=PREVIEW_WIDTH):
    rgb, mask = record["rgb"], record["mask"]
    overlay = overlay_pixels(rgb, mask, opacity)
    # 미리보기 불투명도는 표시만 바꾸며 저장한 overlay.png와 해시는 유지합니다.
    panels = [Image.fromarray(rgb),
              Image.fromarray(mask.astype(np.uint8) * 255).convert("RGB"),
              Image.fromarray(overlay)]
    height = max(1, round(rgb.shape[0] * width / rgb.shape[1]))
    margin, gap, header = 16, 16, 42
    canvas = Image.new("RGB", (margin * 2 + width * 3 + gap * 2, height + header + margin), "#f5f7fa")
    draw = ImageDraw.Draw(canvas)
    font_path = Path("C:/Windows/Fonts/malgun.ttf")
    font = ImageFont.truetype(str(font_path), 19) if font_path.is_file() else ImageFont.load_default()
    local_original = record["metadata"].get("image_source") == "local_original"
    image_title = "원본 이미지" if local_original else "등록 이미지"
    image_title_en = "Original image" if local_original else "Registered image"
    titles = [image_title, "Exposure 마스크", "마스크 오버레이"] if font_path.is_file() else [image_title_en, "Exposure mask", "Overlay"]
    for col, (title, panel) in enumerate(zip(titles, panels)):
        x = margin + col * (width + gap)
        draw.text((x, 8), title, fill="#192b3f", font=font)
        resample = Image.Resampling.NEAREST if col == 1 else Image.Resampling.LANCZOS
        canvas.paste(panel.resize((width, height), resample), (x, header))
    return canvas



def display_record(record, opacity=DEFAULT_OPACITY):
    from IPython.display import display
    m = record["metadata"]
    print(f"Task {m['task_id']} | Annotation {m['annotation_id']} | "
          f"{m['width']}×{m['height']} | Exposure {m['exposure_pixels']:,} pixels")
    print(f"저장: {record['folder']}")
    display(preview_image(record, opacity))



## 3. Project 13 / tab 10의 작업 목록 확인

프로젝트와 tab의 소속을 확인한 뒤 해당 tab에 표시되는 Task ID를 페이지별로 조회합니다. `TASK_IDS = None`이면 전체 목록, `(802, 803)`처럼 지정하면 그 작업만 받습니다. 지정한 ID가 tab에 없으면 중단합니다.


In [ ]:
client = LabelStudioClient(BASE_URL, REFRESH_TOKEN, PROJECT_ID)
project = client.get_json(f"api/projects/{PROJECT_ID}/")
if int(project["id"]) != PROJECT_ID:
    raise RuntimeError("프로젝트 응답 ID가 설정과 다릅니다.")
view = client.view(VIEW_ID)
view_task_ids = tuple(client.list_task_ids(view_id=VIEW_ID))
DOWNLOAD_TASK_IDS = view_task_ids if TASK_IDS is None else tuple(int(value) for value in TASK_IDS)
if len(set(DOWNLOAD_TASK_IDS)) != len(DOWNLOAD_TASK_IDS):
    raise ValueError("TASK_IDS에 중복 ID가 있습니다.")
missing = set(DOWNLOAD_TASK_IDS) - set(view_task_ids)
if missing:
    raise ValueError(f"Project {PROJECT_ID} / tab {VIEW_ID}에 없는 Task ID: {sorted(missing)}")
DOWNLOAD_TASK_IDS = tuple(sorted(DOWNLOAD_TASK_IDS))
print(f"Project {PROJECT_ID}: {project.get('title', '')} / tab {VIEW_ID}: {(view.get('data') or {}).get('title', '')}")
print(f"tab 전체 {len(view_task_ids)}개 · 다운로드 대상 {len(DOWNLOAD_TASK_IDS)}개")
print(f"Task ID: {DOWNLOAD_TASK_IDS}")


## 4. 한 장씩 다운로드 및 저장

이 셀을 실행하면 현재 tab의 작업을 순서대로 `data/gold/task_<ID>/`에 저장합니다. Enter 입력은 필요 없습니다. 이미지와 마스크를 전체 리스트에 쌓지 않고, 한 작업 저장 후 배열 참조를 해제합니다.

일부 작업이 실패해도 나머지는 계속 저장합니다. 결과는 `data/gold/download_summary.json`에 기록하며 마지막에 실패 ID를 알립니다. 재실행하면 같은 Task 폴더를 최신 상태로 갱신합니다. 필요하면 1번 설정 셀의 `TASK_IDS`에 실패 ID만 지정해 다시 받으세요.


In [ ]:
RUN_DOWNLOAD_ALL = True  # False이면 자동 저장하지 않고 아래 선택 셀만 사용할 수 있습니다.
if RUN_DOWNLOAD_ALL:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    downloaded_records = {}  # 작은 metadata만 보관. 이미지/마스크 배열은 보관하지 않습니다.
    download_errors = {}
    def save_download_summary():
        write_json(CACHE_DIR / "download_summary.json", {
            "project_id": PROJECT_ID, "view_id": VIEW_ID, "project_url": PROJECT_URL,
            "requested_task_ids": list(DOWNLOAD_TASK_IDS), "saved_count": len(downloaded_records),
            "completed": len(downloaded_records) == len(DOWNLOAD_TASK_IDS),
            "tasks": list(downloaded_records.values()), "errors": download_errors,
            "updated_at_utc": datetime.now(timezone.utc).isoformat(),
        })
    save_download_summary()
    for index, task_id in enumerate(DOWNLOAD_TASK_IDS, 1):
        record = None
        print(f"[{index}/{len(DOWNLOAD_TASK_IDS)}] Project {PROJECT_ID} · Task {task_id}: 다운로드 중...", flush=True)
        try:
            record = download_one(client, task_id)
            metadata = record["metadata"]
            downloaded_records[task_id] = metadata
            print(f"완료: Task {task_id} | Annotation {metadata['annotation_id']} | "
                  f"{metadata['width']}×{metadata['height']} | Exposure {metadata['exposure_pixels']:,} px | "
                  f"저장: {record['folder']}", flush=True)
        except Exception as error:
            download_errors[task_id] = f"{type(error).__name__}: {error}"
            print(f"실패: Task {task_id} · {download_errors[task_id]}", flush=True)
        finally:
            record = None
            save_download_summary()
    print(f"저장 완료: {len(downloaded_records)}/{len(DOWNLOAD_TASK_IDS)}개 · {CACHE_DIR}")
    if download_errors:
        raise RuntimeError(f"실패한 Task ID: {sorted(download_errors)}. 위 오류와 download_summary.json을 확인하세요.")
else:
    print("전체 다운로드는 꺼져 있습니다. 아래 한 장/한 장씩 확인 셀을 사용하세요.")


## 5. 저장 결과

```text
data/gold/
  download_summary.json
  task_802/
    image.png
    mask.png
    mask.npy
    mask_alpha.png
    mask_alpha.npy
    overlay.png
    annotation.json
    task.json
    metadata.json
  task_<다음 ID>/...
```

NumPy 파일은 `np.load(path, allow_pickle=False)`로 읽습니다. `mask.npy`는 `uint8` 0/1이며 `mask_alpha.npy`는 원래 alpha입니다. `annotation.json`은 해당 이미지의 최신 Annotation 하나만, `task.json`은 API 원본 전체를 담습니다. P1/P2 좌표 생성이나 다른 프로젝트 라벨과의 결합은 하지 않습니다.


## 6. 선택 사항: 새 창으로 한 장씩 보기

`RUN_VIEWER = True`로 바꾸면 새 창에서 Project 13의 Task ID(예: 802)를 입력해 이미지·마스크·오버레이를 확인할 수 있습니다. 불투명도는 표시만 바꿉니다. 기본적으로 창을 열지 않습니다.


In [ ]:
RUN_VIEWER = False
if RUN_VIEWER:
    try:
        import tkinter as tk
        from tkinter import ttk
        from PIL import Image, ImageTk
    except ImportError:
        raise ImportError("현재 노트북 커널에 Tkinter와 Pillow가 필요합니다.") from None

    from queue import Empty, Queue
    from threading import Thread
    from IPython import get_ipython


    class TaskViewer:
        MIN_ZOOM = 0.02
        MAX_ZOOM = 16.0
        ZOOM_STEP = 1.15

        def __init__(self, window):
            self.window = window
            self.client = client
            self.downloader = download_one
            self.records_cache = {}
            self.task_id = None
            self.record = None
            self.busy = False
            self.closed = False
            self.results = Queue()

            self.photo = None
            self.base_image = None
            self.image_item = None
            self.preview_dirty = True
            self.zoom = 1.0
            self.fit_mode = True
            self.redraw_id = None
            self.poll_id = None

            self.task_text = tk.StringVar(window)
            self.opacity = tk.DoubleVar(window, value=DEFAULT_OPACITY)
            self.opacity_text = tk.StringVar(window, value=f"{DEFAULT_OPACITY:.0%}")
            self.zoom_text = tk.StringVar(window, value="배율: -")
            self.status = tk.StringVar(
                window,
                value="Task ID를 입력하고 Enter 또는 불러오기를 누르세요."
            )
            self.details = tk.StringVar(window)

            window.title(f"Label Studio · Project {PROJECT_ID} · 이미지 / 마스크")
            window.geometry("1500x900")
            window.minsize(900, 550)
            window.columnconfigure(0, weight=1)
            window.rowconfigure(4, weight=1)
            window.protocol("WM_DELETE_WINDOW", self.close)

            controls = ttk.Frame(window, padding=(12, 12, 12, 6))
            controls.grid(row=0, column=0, sticky="ew")

            ttk.Label(controls, text="Task ID:").pack(side="left", padx=(0, 6))
            self.entry = ttk.Entry(
                controls, textvariable=self.task_text, width=14
            )
            self.entry.pack(side="left", padx=(0, 8))
            self.entry.bind("<Return>", self.load_task)

            self.load_button = ttk.Button(
                controls, text="불러오기", command=self.load_task
            )
            self.load_button.pack(side="left", padx=(0, 8))

            self.reload_button = ttk.Button(
                controls, text="현재 작업 새로 받기", command=self.reload_task
            )
            self.reload_button.pack(side="left", padx=(0, 20))

            ttk.Label(controls, text="불투명도:").pack(side="left", padx=(0, 6))
            self.slider = ttk.Scale(
                controls,
                from_=0,
                to=1,
                variable=self.opacity,
                command=self.opacity_changed,
                length=180,
            )
            self.slider.pack(side="left")
            ttk.Label(
                controls, textvariable=self.opacity_text, width=5
            ).pack(side="left", padx=6)

            self.fit_button = ttk.Button(
                controls, text="화면 맞춤", command=self.fit_to_window
            )
            self.fit_button.pack(side="left", padx=(16, 6))

            self.actual_button = ttk.Button(
                controls, text="100%", command=lambda: self.set_zoom(1.0)
            )
            self.actual_button.pack(side="left", padx=(0, 8))

            ttk.Label(
                controls, textvariable=self.zoom_text, width=13
            ).pack(side="left")

            ttk.Label(
                window, textvariable=self.status, padding=(12, 6)
            ).grid(row=1, column=0, sticky="w")

            ttk.Label(
                window, textvariable=self.details, padding=(12, 0, 12, 6)
            ).grid(row=2, column=0, sticky="w")

            ttk.Label(
                window,
                text=(
                    "마우스 휠: 커서 위치 확대·축소  |  "
                    "왼쪽 버튼 드래그: 화면 이동  |  "
                    "더블클릭: 화면 맞춤  |  Shift+휠: 가로 이동"
                ),
                padding=(12, 0, 12, 8),
            ).grid(row=3, column=0, sticky="w")

            images = ttk.Frame(window)
            images.grid(row=4, column=0, sticky="nsew", padx=12, pady=(0, 12))
            images.columnconfigure(0, weight=1)
            images.rowconfigure(0, weight=1)

            self.canvas = tk.Canvas(
                images,
                background="#f5f7fa",
                highlightthickness=0,
                borderwidth=0,
                xscrollincrement=1,
                yscrollincrement=1,
            )
            self.canvas.grid(row=0, column=0, sticky="nsew")

            vertical = ttk.Scrollbar(
                images, orient="vertical", command=self.scroll_y
            )
            vertical.grid(row=0, column=1, sticky="ns")

            horizontal = ttk.Scrollbar(
                images, orient="horizontal", command=self.scroll_x
            )
            horizontal.grid(row=1, column=0, sticky="ew")

            self.canvas.configure(
                yscrollcommand=vertical.set,
                xscrollcommand=horizontal.set,
            )

            self.canvas.bind("<Configure>", self.on_resize)
            self.canvas.bind("<MouseWheel>", self.mouse_wheel)
            self.canvas.bind("<Button-4>", self.mouse_wheel)
            self.canvas.bind("<Button-5>", self.mouse_wheel)

            self.canvas.bind("<ButtonPress-1>", self.start_pan)
            self.canvas.bind("<B1-Motion>", self.pan)
            self.canvas.bind("<ButtonRelease-1>", self.end_pan)
            self.canvas.bind("<Double-Button-1>", self.fit_to_window)

            self.canvas.create_text(
                24,
                24,
                anchor="nw",
                text="Task ID를 입력하면 이미지와 마스크가 여기에 표시됩니다.",
                fill="#192b3f",
                font=("Malgun Gothic", 12),
            )

            self.set_busy(False)
            self.entry.focus_set()
            self.poll_id = window.after(100, self.poll_results)

        def set_busy(self, busy):
            self.busy = busy
            state = "disabled" if busy else "normal"

            self.entry.configure(state=state)
            self.load_button.configure(state=state)
            self.reload_button.configure(
                state="disabled" if busy or self.task_id is None else "normal"
            )

            image_state = (
                "disabled" if busy or self.record is None else "normal"
            )
            self.slider.configure(state=image_state)
            self.fit_button.configure(state=image_state)
            self.actual_button.configure(state=image_state)

        def load_task(self, event=None):
            if self.busy or self.closed:
                return
            try:
                task_id = int(self.task_text.get().strip())
                if task_id <= 0:
                    raise ValueError
            except ValueError:
                self.status.set("Task ID에 양의 정수를 입력하세요. 예: 802")
                return
            self.load(task_id)

        def reload_task(self):
            if not self.busy and not self.closed and self.task_id is not None:
                self.load(self.task_id, force=True)

        def load(self, task_id, force=False):
            self.task_id = task_id
            self.task_text.set(str(task_id))
            self.record = None
            self.base_image = None
            self.photo = None
            self.image_item = None
            self.preview_dirty = True
            self.fit_mode = True
            self.details.set("")
            self.zoom_text.set("배율: -")

            self.canvas.delete("all")
            self.canvas.configure(scrollregion=(0, 0, 0, 0))
            self.canvas.xview_moveto(0)
            self.canvas.yview_moveto(0)
            self.set_busy(True)

            if not force and task_id in self.records_cache:
                self.show_record(task_id, self.records_cache[task_id])
                return

            self.status.set(f"Task {task_id} 이미지와 최신 마스크를 받는 중...")
            Thread(target=self.download, args=(task_id,), daemon=True).start()

        def download(self, task_id):
            # 다운로드만 별도 스레드에서 수행합니다.
            try:
                record = self.downloader(self.client, task_id)
                self.results.put((task_id, record, None))
            except Exception as error:
                self.results.put(
                    (task_id, None, f"{type(error).__name__}: {error}")
                )

        def poll_results(self):
            if self.closed:
                return

            try:
                task_id, record, error = self.results.get_nowait()
            except Empty:
                pass
            else:
                if error is None:
                    self.records_cache.clear()  # 현재 한 장만 유지합니다.
                    self.records_cache[task_id] = record
                    self.show_record(task_id, record)
                else:
                    self.status.set(f"Task {task_id} 실패 · {error}")
                    self.set_busy(False)

            self.poll_id = self.window.after(100, self.poll_results)

        def show_record(self, task_id, record):
            try:
                self.record = record
                self.preview_dirty = True
                self.fit_mode = True

                metadata = record["metadata"]
                self.details.set(
                    f"Annotation {metadata['annotation_id']} · "
                    f"{metadata['width']}×{metadata['height']} · "
                    f"{LABEL_NAME} {metadata['exposure_pixels']:,} pixels\n"
                    f"저장: {record['folder']}"
                )
                self.redraw()
                self.status.set(f"Project {PROJECT_ID} · Task {task_id}")
            except Exception as error:
                self.record = None
                self.status.set(
                    f"Task {task_id} 표시 실패 · "
                    f"{type(error).__name__}: {error}"
                )
            finally:
                self.set_busy(False)

        def build_preview(self):
            if self.record is None:
                return

            # 원본 해상도로 구성해 확대할 때 세부 정보가 유지되게 합니다.
            # 휠을 돌릴 때마다 재생성하지 않고 불투명도가 바뀔 때만 갱신합니다.
            width = int(self.record["rgb"].shape[1])
            self.base_image = preview_image(
                self.record,
                opacity=float(self.opacity.get()),
                width=width,
            ).convert("RGB")
            self.preview_dirty = False

        def update_scrollregion(self):
            if self.base_image is None:
                return

            self.region_width = max(
                self.canvas.winfo_width(),
                round(self.base_image.width * self.zoom),
            )
            self.region_height = max(
                self.canvas.winfo_height(),
                round(self.base_image.height * self.zoom),
            )
            self.canvas.configure(
                scrollregion=(0, 0, self.region_width, self.region_height)
            )
            self.zoom_text.set(f"배율: {self.zoom * 100:.1f}%")

        def fit_to_window(self, event=None):
            if self.record is None or self.closed:
                return "break"

            self.fit_mode = True
            self.schedule_redraw(delay=0)
            return "break"

        def set_zoom(self, value, x=None, y=None):
            if self.base_image is None or self.closed:
                return "break"

            width = max(1, self.canvas.winfo_width())
            height = max(1, self.canvas.winfo_height())
            x = width / 2 if x is None else x
            y = height / 2 if y is None else y

            new_zoom = max(self.MIN_ZOOM, min(self.MAX_ZOOM, float(value)))
            if abs(new_zoom - self.zoom) < 1e-10:
                return "break"

            # 확대 전 커서 아래의 원본 좌표를 계산합니다.
            source_x = self.canvas.canvasx(x) / self.zoom
            source_y = self.canvas.canvasy(y) / self.zoom

            self.zoom = new_zoom
            self.fit_mode = False
            self.update_scrollregion()

            # 확대 후에도 같은 원본 위치가 커서 아래에 오도록 이동합니다.
            left = source_x * self.zoom - x
            top = source_y * self.zoom - y
            left = max(0, min(left, self.region_width - width))
            top = max(0, min(top, self.region_height - height))

            self.canvas.xview_moveto(left / self.region_width)
            self.canvas.yview_moveto(top / self.region_height)
            self.schedule_redraw(delay=16)
            return "break"

        def mouse_wheel(self, event):
            if self.base_image is None or self.closed:
                return "break"

            button = getattr(event, "num", None)
            delta = getattr(event, "delta", 0)

            if button == 4:
                steps = 1
            elif button == 5:
                steps = -1
            elif delta:
                steps = delta / 120 if abs(delta) >= 120 else (1 if delta > 0 else -1)
                steps = max(-3, min(3, steps))
            else:
                return "break"

            # Shift + 휠은 가로 스크롤입니다.
            if event.state & 0x0001:
                self.canvas.xview_scroll(round(-steps * 60), "units")
                self.schedule_redraw(delay=16)
                return "break"

            return self.set_zoom(
                self.zoom * self.ZOOM_STEP ** steps,
                x=event.x,
                y=event.y,
            )

        def start_pan(self, event):
            if self.base_image is not None:
                self.canvas.scan_mark(event.x, event.y)
                self.canvas.configure(cursor="fleur")

        def pan(self, event):
            if self.base_image is None:
                return
            self.canvas.scan_dragto(event.x, event.y, gain=1)
            self.schedule_redraw(delay=16)

        def end_pan(self, event):
            self.canvas.configure(cursor="")

        def scroll_x(self, *args):
            self.canvas.xview(*args)
            self.schedule_redraw(delay=16)

        def scroll_y(self, *args):
            self.canvas.yview(*args)
            self.schedule_redraw(delay=16)

        def on_resize(self, event=None):
            # 수동 확대 중에는 창 크기가 바뀌어도 배율을 유지합니다.
            self.schedule_redraw(delay=80)

        def redraw(self):
            if self.closed or self.record is None:
                return

            if self.preview_dirty or self.base_image is None:
                self.build_preview()

            width = max(1, self.canvas.winfo_width())
            height = max(1, self.canvas.winfo_height())

            if self.fit_mode:
                self.zoom = max(
                    self.MIN_ZOOM,
                    min(
                        width / self.base_image.width,
                        height / self.base_image.height,
                        1.0,
                    ),
                )
                self.update_scrollregion()
                self.canvas.xview_moveto(0)
                self.canvas.yview_moveto(0)
            else:
                self.update_scrollregion()

            left = self.canvas.canvasx(0)
            top = self.canvas.canvasy(0)

            # 전체 이미지를 거대한 크기로 확대하지 않고,
            # 현재 화면에 보이는 부분만 렌더링합니다.
            source_box = (
                left / self.zoom,
                top / self.zoom,
                (left + width) / self.zoom,
                (top + height) / self.zoom,
            )
            resample = (
                Image.Resampling.NEAREST
                if self.zoom >= 1.0
                else Image.Resampling.BILINEAR
            )
            visible = self.base_image.transform(
                (width, height),
                Image.Transform.EXTENT,
                source_box,
                resample=resample,
                fillcolor="#f5f7fa",
            )
            photo = ImageTk.PhotoImage(visible, master=self.window)

            if self.image_item is None:
                self.image_item = self.canvas.create_image(
                    left, top, anchor="nw", image=photo
                )
            else:
                self.canvas.coords(self.image_item, left, top)
                self.canvas.itemconfigure(self.image_item, image=photo)

            self.photo = photo

        def redraw_later(self):
            self.redraw_id = None
            try:
                self.redraw()
            except Exception as error:
                self.status.set(
                    f"미리보기 갱신 실패 · {type(error).__name__}: {error}"
                )

        def schedule_redraw(self, event=None, delay=120):
            if self.closed:
                return
            if self.redraw_id is not None:
                self.window.after_cancel(self.redraw_id)
            self.redraw_id = self.window.after(delay, self.redraw_later)

        def opacity_changed(self, value):
            self.opacity_text.set(f"{float(value):.0%}")
            self.preview_dirty = True
            self.schedule_redraw(delay=120)

        def close(self):
            if self.closed:
                return
            self.closed = True

            for callback_id in (self.poll_id, self.redraw_id):
                if callback_id is not None:
                    try:
                        self.window.after_cancel(callback_id)
                    except tk.TclError:
                        pass

            self.window.destroy()


    # 노트북에서 Tk 창이 셀 실행 후에도 반응하도록 이벤트 루프를 연결합니다.
    notebook_shell = get_ipython()
    if notebook_shell is not None:
        notebook_shell.run_line_magic("gui", "tk")

    if "viewer" in globals():
        try:
            viewer.close()
        except tk.TclError:
            pass

    viewer_window = tk.Tk()
    viewer = TaskViewer(viewer_window)

    print(
        "Task ID 입력 → 불러오기 | "
        "휠: 확대·축소 | 드래그: 이동 | 더블클릭: 화면 맞춤"
    )

    if notebook_shell is None:
        viewer_window.mainloop()

## 7. 선택 사항: 특정 한 장 / Enter로 한 장씩 저장

`RUN_SINGLE = True`이면 대상 목록의 첫 작업 한 장을 저장하고 미리보기를 표시합니다. `RUN_STEP_LOOP = True`이면 대상 목록을 한 장씩 저장·표시한 다음 Enter로 이동하며 `q`로 종료합니다. 이 방식을 쓸 때 4절의 `RUN_DOWNLOAD_ALL = False`로 두면 전체 다운로드를 먼저 실행하지 않습니다.


In [ ]:
RUN_SINGLE = False
if RUN_SINGLE:
    if not DOWNLOAD_TASK_IDS:
        raise RuntimeError("다운로드 대상 작업이 없습니다.")
    record = download_one(client, DOWNLOAD_TASK_IDS[0])
    display_record(record, opacity=DEFAULT_OPACITY)


In [ ]:
RUN_STEP_LOOP = False
if RUN_STEP_LOOP:
    from IPython.display import clear_output
    for task_id in DOWNLOAD_TASK_IDS:
        clear_output(wait=True)
        try:
            record = download_one(client, task_id)
            display_record(record, opacity=DEFAULT_OPACITY)
        except Exception as error:
            print(f"Task {task_id} 실패: {type(error).__name__}: {error}")
        if input("Enter: 다음 한 장 / q: 종료 > ").strip().lower() == "q":
            break
    

## 참고

- 저장 위치는 `data/gold/task_<ID>/`입니다. 서버 이미지와 원래 Annotation 좌표를 사용합니다.
- latest Annotation을 고를 때 취소된 Annotation을 제외하고 수정 시각과 ID로 순서를 결정합니다. 빈 완료 Annotation은 빈 마스크로 보존하며, Annotation 자체가 없으면 실패로 기록합니다.
- 이미지·마스크 크기나 image_rotation이 맞지 않으면 임의 리사이즈·회전 없이 해당 작업을 실패 처리합니다.
- `tab=10`은 [작업 목록 API의 view 인자](https://api.labelstud.io/api-reference/api-reference/tasks/list)에 전달합니다. tab 필터를 무시하고 프로젝트 전체로 확장하지 않습니다.
- 기존 학습 데이터나 체크포인트를 바꾸지 않습니다. 인증은 [Label Studio access token 문서](https://labelstud.io/guide/access_tokens), 작업 조회는 [Get task API](https://api.labelstud.io/api-reference/api-reference/tasks/get)를 참고하세요.
